<a href="https://colab.research.google.com/github/eric20041027/Data_Mining/blob/main/notebooks/train_pubmedbert_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# === 0. 環境設定：clone repo、裝套件、掛 Drive、確認 GPU ===
import os, sys, shutil

REPO_DIR = '/content/Data_Mining'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/eric20041027/Data_Mining.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# 掛 Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU 確認
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('bf16 supported:', torch.cuda.is_bf16_supported())

# 安裝套件
!pip install -q -U "transformers>=4.44,<4.50" "accelerate>=0.33" "datasets>=2.20" "scikit-learn>=1.4"
print('\nSetup OK.')

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.98 KiB | 2.98 MiB/s, done.
From https://github.com/eric20041027/Data_Mining
   030efb1..1d3827d  main       -> origin/main
Updating 030efb1..1d3827d
Fast-forward
 notebooks/train_pubmedbert_colab.ipynb | 1753 ++++----------------------------
 1 file changed, 176 insertions(+), 1577 deletions(-)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA: True
Device: NVIDIA A100-SXM4-40GB
bf16 supported: True

Setup OK.


In [19]:
# === 1. 從 Drive 還原舊的 prediction artifacts (val_probs/test_probs/args)===
import tarfile, os

candidates = [
    '/content/drive/MyDrive/Kaggle_backup/predictions_only_after_noweight.tar.gz',
    '/content/drive/MyDrive/Kaggle_backup/predictions_only_before_noweight.tar.gz',
    '/content/drive/MyDrive/Kaggle_backup/predictions_only.tar.gz',
]
for tar_path in candidates:
    if os.path.exists(tar_path):
        print(f'Restoring from: {tar_path}')
        with tarfile.open(tar_path) as tar:
            tar.extractall('/content/Data_Mining')
        print('  done')
        break

# 檢查還原狀態
src = '/content/Data_Mining/outputs/bert_runs'
if os.path.isdir(src):
    dirs = sorted(d for d in os.listdir(src) if os.path.isdir(os.path.join(src, d)))
    print(f'\n=== 目前 {len(dirs)} 個 run 資料夾 ===')
    for d in dirs:
        print(f'  {d}')
else:
    print('outputs/bert_runs 不存在，將從零開始訓練')

Restoring from: /content/drive/MyDrive/Kaggle_backup/predictions_only_after_noweight.tar.gz
  done

=== 目前 21 個 run 資料夾 ===
  biobert_base_seed42_fold0
  biobert_base_seed42_fold1
  biobert_base_seed42_fold2
  biobert_base_seed42_fold3
  biobert_base_seed42_fold4
  pubmedbert_base_seed2024_fold0
  pubmedbert_base_seed2024_fold1
  pubmedbert_base_seed2024_fold2
  pubmedbert_base_seed2024_fold3
  pubmedbert_base_seed2024_fold4
  pubmedbert_base_seed42_fold0
  pubmedbert_base_seed42_fold1
  pubmedbert_base_seed42_fold2
  pubmedbert_base_seed42_fold3
  pubmedbert_base_seed42_fold4
  pubmedbert_noweight_seed42_fold0
  pubmedbert_noweight_seed42_fold1
  pubmedbert_noweight_seed42_fold2
  pubmedbert_noweight_seed42_fold3
  pubmedbert_noweight_seed42_fold4
  smoke_test


/tmp/ipykernel_1032/3222745984.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content/Data_Mining')


In [20]:
# === 2. 共用工具：訓練、備份、ensemble ===
import os, subprocess, tarfile, glob, json
import pandas as pd

def train_model(model, seed, epochs, bs, lr, tag_prefix, class_weight='none'):
    """訓練單一模型 5-fold"""
    for fold in range(5):
        cmd = (
            f'python src/train_bert.py --model {model} '
            f'--fold {fold} --seed {seed} --epochs {epochs} --batch-size {bs} '
            f'--lr {lr} --max-length 512 --class-weight {class_weight} '
            f'--tag {tag_prefix}_fold{fold}'
        )
        print('>>>', cmd)
        rc = os.system(cmd)
        assert rc == 0, f'{tag_prefix} fold {fold} failed'

    # 印 OOF 摘要
    runs = sorted(glob.glob(f'outputs/bert_runs/{tag_prefix}_fold*/metrics.json'))
    df = pd.DataFrame([json.load(open(p)) for p in runs])
    print(f'\n{tag_prefix} per-fold val Macro F1:')
    print(df[['fold', 'val_macro_f1', 'train_secs']])
    print(f'{tag_prefix} mean OOF Macro F1: {df.val_macro_f1.mean():.4f}\n')

def backup(label):
    """輕量備份 (~50MB)：只打包 .npy/.json/.csv"""
    src = '/content/Data_Mining/outputs/bert_runs'
    dst = f'/content/drive/MyDrive/Kaggle_backup/predictions_only_{label}.tar.gz'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    files_to_pack = []
    for run in sorted(os.listdir(src)):
        rd = os.path.join(src, run)
        if not os.path.isdir(rd):
            continue
        for pat in ['*.npy', '*.json', '*.csv']:
            files_to_pack += glob.glob(os.path.join(rd, pat))
    with tarfile.open(dst, 'w:gz') as tar:
        for f in files_to_pack:
            tar.add(f, arcname=os.path.relpath(f, '/content/Data_Mining'))
    size = subprocess.check_output(['du', '-h', dst]).decode().split()[0]
    print(f'Backup: {dst} ({size})')

def run_ensemble(patterns, tag):
    """跑 ensemble 並印關鍵指標"""
    cmd = ['python', 'src/ensemble_predict.py'] + ['--bert-runs'] + patterns + ['--no-overlap-constraint', '--tag', tag]
    print('>>>', ' '.join(cmd))
    out = subprocess.run(cmd, cwd='/content/Data_Mining', capture_output=True, text=True)
    if out.returncode != 0:
        print('STDERR:', out.stderr)
        return
    lines = out.stdout.split('\n')
    for line in lines:
        if any(k in line for k in ['Found ', 'fold ', 'OOF Macro F1', 'macro avg', 'general path']):
            print(line)
    print()

print('工具函數已載入。')

工具函數已載入。


In [21]:
# === 3. BioBERT noweight 5-fold (~50 分鐘) ===
train_model(
    model='dmis-lab/biobert-base-cased-v1.2',
    seed=42, epochs=4, bs=32, lr=2e-5,
    tag_prefix='biobert_noweight_seed42',
    class_weight='none'
)
backup('after_biobert_noweight')

>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 0 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag biobert_noweight_seed42_fold0
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 1 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag biobert_noweight_seed42_fold1
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 2 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag biobert_noweight_seed42_fold2
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 3 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag biobert_noweight_seed42_fold3
>>> python src/train_bert.py --model dmis-lab/biobert-base-cased-v1.2 --fold 4 --seed 42 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag biobert_noweight_seed42_fold4


In [22]:
# === 4. PubMedBERT noweight seed=2024 5-fold (~50 分鐘) ===
train_model(
    model='microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
    seed=2024, epochs=4, bs=32, lr=2e-5,
    tag_prefix='pubmedbert_noweight_seed2024',
    class_weight='none'
)
backup('after_pubmedbert_noweight_2024')

>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 0 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed2024_fold0
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 1 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed2024_fold1
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 2 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed2024_fold2
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext --fold 3 --seed 2024 --epochs 4 --batch-size 32 --lr 2e-05 --max-length 512 --class-weight none --tag pubmedbert_noweight_seed2024_fold3
>>> python src/train_bert.py --model microsoft/BiomedNLP

In [23]:
# === 5. 中途 ensemble，看 3 個 noweight 模型合起來表現 ===
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
], 'v12_3noweight')

>>> python src/ensemble_predict.py --bert-runs outputs/bert_runs/pubmedbert_noweight_seed*_fold* outputs/bert_runs/biobert_noweight_seed*_fold* --no-overlap-constraint --tag v12_3noweight
Found 15 BERT run dirs:
  fold 0: 3 run(s), val macro F1 = 0.6696
  fold 1: 3 run(s), val macro F1 = 0.6600
  fold 2: 3 run(s), val macro F1 = 0.6465
  fold 3: 3 run(s), val macro F1 = 0.6693
  fold 4: 3 run(s), val macro F1 = 0.6363
BERT OOF Macro F1: 0.6566
general pathological conditions     0.6605    0.4179    0.5119      4334
                      macro avg     0.6441    0.6879    0.6566     12994
Ensemble OOF Macro F1: 0.6566
general pathological conditions     0.6605    0.4179    0.5119      4334
                      macro avg     0.6441    0.6879    0.6566     12994



In [26]:
from google.colab import files
files.download('/content/Data_Mining/outputs/submission_v12_3noweight.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
# === 6. PubMedBERT-large noweight (~80 分鐘) ===
train_model(
    model='microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract',
    seed=42, epochs=3, bs=16, lr=1e-5,
    tag_prefix='pubmedbertlarge_noweight_seed42',
    class_weight='none'
)
backup('final_all_noweight')

>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract --fold 0 --seed 42 --epochs 3 --batch-size 16 --lr 1e-05 --max-length 512 --class-weight none --tag pubmedbertlarge_noweight_seed42_fold0
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract --fold 1 --seed 42 --epochs 3 --batch-size 16 --lr 1e-05 --max-length 512 --class-weight none --tag pubmedbertlarge_noweight_seed42_fold1
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract --fold 2 --seed 42 --epochs 3 --batch-size 16 --lr 1e-05 --max-length 512 --class-weight none --tag pubmedbertlarge_noweight_seed42_fold2
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract --fold 3 --seed 42 --epochs 3 --batch-size 16 --lr 1e-05 --max-length 512 --class-weight none --tag pubmedbertlarge_noweight_seed42_fold3
>>> python src/train_bert.py --model microsoft/BiomedNLP-BiomedBERT-large-uncased-ab

In [25]:
# === 7. 全部 noweight 模型 ensemble，產生最終 submissions ===

# 候選 A: 純 noweight (seed=42)
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
], 'final_a_noweight42')

# 候選 B: 2 個 PubMedBERT noweight seeds
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
], 'final_b_pubmed_2seeds')

# 候選 C: PubMedBERT × 2 + BioBERT (3 個 base noweight)
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
], 'final_c_3noweight')

# 候選 D: 全部 4 個 noweight 模型（含 large）
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
], 'final_d_4noweight')

# 顯示所有 final submission
print('\n=== 所有 final submissions ===')
import os, hashlib, pandas as pd
for tag in ['final_a_noweight42', 'final_b_pubmed_2seeds', 'final_c_3noweight', 'final_d_4noweight']:
    p = f'outputs/submission_{tag}.csv'
    if os.path.exists(p):
        h = hashlib.md5(open(p, 'rb').read()).hexdigest()[:10]
        df = pd.read_csv(p)
        dist = df['label'].value_counts(normalize=True).sort_index().round(3).to_dict()
        print(f'  {tag}: md5={h} dist={dist}')

# 下載
from google.colab import files
for tag in ['final_a_noweight42', 'final_b_pubmed_2seeds', 'final_c_3noweight', 'final_d_4noweight']:
    p = f'/content/Data_Mining/outputs/submission_{tag}.csv'
    if os.path.exists(p):
        files.download(p)

print('\n=== 全部完成 ===')

>>> python src/ensemble_predict.py --bert-runs outputs/bert_runs/pubmedbert_noweight_seed42_fold* --no-overlap-constraint --tag final_a_noweight42
Found 5 BERT run dirs:
  fold 0: 1 run(s), val macro F1 = 0.6609
  fold 1: 1 run(s), val macro F1 = 0.6560
  fold 2: 1 run(s), val macro F1 = 0.6501
  fold 3: 1 run(s), val macro F1 = 0.6596
  fold 4: 1 run(s), val macro F1 = 0.6333
BERT OOF Macro F1: 0.6524
general pathological conditions     0.6654    0.3964    0.4968      4334
                      macro avg     0.6390    0.6892    0.6524     12994
Ensemble OOF Macro F1: 0.6524
general pathological conditions     0.6654    0.3964    0.4968      4334
                      macro avg     0.6390    0.6892    0.6524     12994

>>> python src/ensemble_predict.py --bert-runs outputs/bert_runs/pubmedbert_noweight_seed*_fold* --no-overlap-constraint --tag final_b_pubmed_2seeds
Found 10 BERT run dirs:
  fold 0: 2 run(s), val macro F1 = 0.6613
  fold 1: 2 run(s), val macro F1 = 0.6566
  fold 2: 2 ru

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== 全部完成 ===
